## Exercise 15.3 (Visualize failure patterns with plotly)  

1. Create a parallel coordinates plot to explore relationships between multiple features and failure types.
2. Use dropdown widgets to filter the plot by failure type and observe how feature relationships change.
3. For each failure type, use the brushing feature (select a range on an axis) to isolate the failing samples and read off approximate thresholds of the features involved. Note down these thresholds; you will need them in Milestone 4.
4. Document any new patterns or anomalies discovered through the interactive exploration.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

df = pd.read_csv("XAI_Drilling_Dataset.csv")

df.head()

,ID,Cutting speed vc [m/min],Spindle speed n [1/min],Feed f [mm/rev],Feed rate vf [mm/min],Power Pc [kW],Cooling [%],Material,Drill Bit Type,Process Time [sec],Main Failure,BEF,CCF,FWF,WDF
0,1,29.92,794,0.218,173,194.87,75,N,W,19.26,0,0,0,0,0
1,2,29.85,792,0.241,191,214.71,75,N,W,19.55,0,0,0,0,0
2,3,22.85,606,0.183,111,95.68,50,P,N,36.16,0,0,0,0,0
3,4,29.60,785,0.236,237,206.93,75,N,W,21.22,1,0,0,1,0
4,5,17.43,462,0.246,114,74.85,50,K,H,33.56,0,0,0,0,0


1. Create a parallel coordinates plot to explore relationships between multiple features and failure types.

In [2]:
# Definiere Einflussgrößen und Zielgrößen
numeric_features = [
    "Cutting speed vc [m/min]",
    "Spindle speed n [1/min]",
    "Feed f [mm/rev]",
    "Feed rate vf [mm/min]",
    "Power Pc [kW]",
    "Cooling [%]",
    "Process Time [sec]"
]

failure_cols = ["BEF", "CCF", "FWF", "WDF"]

# Definiere die Kategorien für die Material- und Bohrertyp-Spalten
df["Material"] = pd.Categorical(
    df["Material"],
    categories=["K", "N", "P"]
)

df["Drill Bit Type"] = pd.Categorical(
    df["Drill Bit Type"],
    categories=["H", "N", "W"]
)

In [ ]:
# Definiere die Funktion zum Erstellen des Parallelkoordinatendiagramms
def plot_parallel(failure):
    dimensions = []

    # Füge die numerischen Features hinzu
    for feature in numeric_features:
        dimensions.append({
            "label": feature,
            "values": df[feature]
        })

    # Füge die kategorischen Features hinzu
    dimensions.append({
        "label": "Material",
        "values": df["Material"].cat.codes,
        "tickvals": [0, 1, 2],
        "ticktext": ["K", "N", "P"],
        "range": [0, 2]
    })

    # Füge die Bohrertyp-Spalte hinzu
    dimensions.append({
        "label": "Drill Bit Type",
        "values": df["Drill Bit Type"].cat.codes,
        "tickvals": [0, 1, 2],
        "ticktext": ["H", "N", "W"],
        "range": [0, 2]
    })

    # Füge die Zielgröße (Fehlerart) hinzu
    dimensions.append({
        "label": failure,
        "values": df[failure],
        "tickvals": [0, 1],
        "ticktext": ["kein Fehler", "Fehler"],
        "range": [0, 1]
    })

    # Erstelle das Parallelkoordinatendiagramm
    fig = go.Figure(
        go.Parcoords(
            dimensions=dimensions,
            line={
                "color": df[failure],
                "cmin": 0,
                "cmax": 1,
                "colorscale": [
                    [0.0, "steelblue"],
                    [0.5, "steelblue"],
                    [0.5, "red"],
                    [1.0, "red"]
                ],
                "showscale": False
            }
        )
    )

    # Passe das Layout des Diagramms an
    fig.update_layout(
        title=f"Parallelkoordinatendiagramm: {failure}",
        height=600,
        margin=dict(l=70, r=70, t=100, b=50)
    )

    fig.show()

# Erstelle das Dropdown-Menü und die interaktive Ausgabe
dropdown = widgets.Dropdown(
    options=failure_cols,
    value="BEF",
    description="Fehlerart:"
)

# Erstelle die interaktive Ausgabe basierend auf der Dropdown-Auswahl
output = widgets.interactive_output(
    plot_parallel,
    {"failure": dropdown}
)

display(dropdown, output)

Dropdown(description='Fehlerart:', options=('BEF', 'CCF', 'FWF', 'WDF'), value='BEF')

Output()

2. Use dropdown widgets to filter the plot by failure type and observe how feature relationships change.

Über das Dropdown wurden nacheinander BEF, CCF, FWF und WDF ausgewählt. Durch Markieren von „Fehler“ auf der jeweiligen Fehlerachse wurden die zugehörigen Beobachtungen hervorgehoben und miteinander verglichen.

Bei **BEF** konzentrieren sich die Fehlerfälle auf geringe Kühlung und niedrige Schnittgeschwindigkeiten in den Materialgruppen K und P. **CCF** zeigt ein breiteres Muster: Viele Fehlerfälle treten bei geringer Kühlung auf, jedoch nicht alle. Die Fehlerart lässt sich deshalb nicht durch die Kühlung allein erklären.

Bei **FWF** ist eine deutliche Häufung bei hoher Vorschubgeschwindigkeit und Material N erkennbar. Einzelne Fehlerfälle gehören zu anderen Materialgruppen. Bei **WDF** ergibt sich dagegen keine klare Trennung anhand der numerischen Messgrößen. Hier fallen die unpassenden Verbindungen zwischen Material und Bohrertyp auf.

Der Vergleich bestätigt die bisherigen Beobachtungen aus Exercise 15.2 und macht zusätzlich die Material-Bohrer-Kombinationen sichtbar.

3. For each failure type, use the brushing feature (select a range on an axis) to isolate the failing samples and read off approximate thresholds of the features involved. Note down these thresholds; you will need them in Milestone 4.

1. Für **BEF** wurden die Fehlerfälle markiert und anschließend nach Material getrennt betrachtet. Bei Material K liegen die Schnittgeschwindigkeiten bis ungefähr 16,7 m/min, bei Material P bis ungefähr 22 m/min. In beiden Gruppen treten die Fehlerfälle bei 0 oder 25 % Kühlung auf. Für Material N werden keine BEF-Fälle angezeigt.
Als mögliche Grenzen werden daher für K eine Schnittgeschwindigkeit von etwa 16,7 m/min und für P etwa 22 m/min, jeweils zusammen mit einer Kühlung von höchstens 25 %, festgehalten. Diese Werte beschreiben zunächst die beobachteten Fehlerbereiche. Ihre Eignung als Vorhersageregel muss später anhand der Beobachtungen ohne BEF geprüft werden.

2. Bei **CCF** wurden zunächst Material K und der dazu passende Bohrertyp H ausgewählt. Die verbleibenden Fehlerfälle zeigen Kühlwerte von 0 oder 25 % sowie erhöhte Vorschubgeschwindigkeiten. Deren unterer Rand liegt im Diagramm ungefähr bei 125–130 mm/min.
Diese Beobachtung passt zur Kombination aus hoher Vorschubgeschwindigkeit und unzureichender Kühlung. Die Werte werden als vorläufige Anhaltspunkte festgehalten. Da nur Fehlerfälle markiert sind, ist damit noch keine eindeutige Trennung von Beobachtungen ohne CCF nachgewiesen.
Bei Material P und dem passenden Bohrertyp N treten die markierten CCF-Fälle bei 0 oder 25 % Kühlung auf. Die Vorschubgeschwindigkeit beginnt ungefähr im Bereich von 120–125 mm/min. Wie bei Material K ist somit eine Kombination aus geringer Kühlung und erhöhter Vorschubgeschwindigkeit erkennbar. Die abgelesenen Werte sind vorläufige Schwellenbereiche und noch keine bestätigten Fehlerregeln.
Bei Material N und dem passenden Bohrertyp W werden keine CCF-Fälle hervorgehoben. Die zuvor festgestellten 36 CCF-Fälle bei Material N treten daher mit anderen Bohrertypen auf. Für die Kombination N–W lässt sich aus den Fehlerfällen keine Vorschub- oder Kühlgrenze ableiten, da diese Auswahl leer ist.
Bei Material N treten die markierten CCF-Fälle mit den ungeeigneten Bohrertypen H oder N auf. Ihre Vorschubgeschwindigkeiten liegen ungefähr zwischen 205 und 245 mm/min, während die Kühlung 75 oder 100 % beträgt. Die Beobachtung ist damit mit der Kombination aus hoher Vorschubgeschwindigkeit und ungeeignetem Bohrer vereinbar.
Zusammen mit den Untersuchungen für K und P zeigt sich, dass CCF durch unterschiedliche Merkmalskombinationen gekennzeichnet ist. Eine einzige Kühlgrenze kann daher nicht alle CCF-Fälle beschreiben. Die abgelesenen Vorschubbereiche werden als vorläufige Hinweise für die spätere Regelbildung festgehalten.

3. Bei **FWF** wurden für Material N die Fehler- und Materialachse isoliert. Sie liegen bei Schnittgeschwindigkeiten von ungefähr 29,1–29,7 m/min und Vorschubgeschwindigkeiten von ungefähr 206–250 mm/min. Damit ist eine Kombination aus vergleichsweise niedriger Schnittgeschwindigkeit innerhalb der Materialgruppe und hoher Vorschubgeschwindigkeit erkennbar.
Als Anhaltspunkte für eine spätere Regel werden eine obere Schnittgeschwindigkeitsgrenze von etwa 29,7 m/min und ein unterer Vorschubbereich um 205–206 mm/min notiert.
Die getrennte Auswahl der Materialien zeigt unterschiedliche FWF-Bereiche. Bei Material K liegen die Fehlerfälle ungefähr bei 16,6–16,7 m/min Schnittgeschwindigkeit und 149–166 mm/min Vorschubgeschwindigkeit. Bei Material P liegen sie ungefähr bei 22 m/min und 127–152 mm/min. Zusammen mit den Ergebnissen für Material N wird deutlich, dass die Materialgruppen getrennt betrachtet werden sollten.
Als vorläufige Anhaltspunkte werden obere Schnittgeschwindigkeiten von etwa 16,7 m/min für K, 22 m/min für P und 29,7 m/min für N notiert. Die unteren beobachteten Vorschubwerte liegen ungefähr bei 149, 127 beziehungsweise 206 mm/min. Diese Werte sind noch keine bestätigten Entscheidungsgrenzen. Insbesondere bei K und P ist die Aussagekraft wegen der wenigen Fehlerfälle begrenzt.

4. Bei **WDF** mit der Auswahl von Material K und WDF = 1 führen die hervorgehobenen Linien zu den Bohrertypen N und W. Der für K passende Bohrertyp H ist unter den ausgewählten Fehlerfällen nicht vertreten. Damit ist für diese Materialgruppe eine unpassende Material-Bohrer-Kombination als Merkmal der WDF-Fälle erkennbar.
Die getrennte Betrachtung der Materialien zeigt bei WDF durchgehend unpassende Material-Bohrer-Kombinationen. Bei Material K führen die markierten Fehlerlinien zu N oder W, bei Material P zu H oder W und bei Material N zu H oder N. Die passenden Kombinationen K–H, P–N und N–W sind unter den ausgewählten WDF-Fällen nicht vertreten.

4. Document any new patterns or anomalies discovered through the interactive exploration.

### Neue Muster und Auffälligkeiten

- **BEF:** Die Fehlerbereiche unterscheiden sich nach Material. Bei K liegen die Schnittgeschwindigkeiten bis ungefähr 16,7 m/min, bei P bis ungefähr 22 m/min, jeweils bei 0 oder 25 % Kühlung. Bei Material N wurden keine BEF-Fälle beobachtet.
- **CCF:** Es zeigen sich unterschiedliche Merkmalskombinationen. Bei K und P treten Fehler auch mit passendem Bohrer bei geringer Kühlung und erhöhtem Vorschub auf. Bei N sind die Fehlerfälle mit ungeeigneten Bohrern und hohen Vorschubgeschwindigkeiten verbunden, obwohl die Kühlung hoch ist.
- **FWF:** Die Fehlerfälle konzentrieren sich auf Material N. Die wenigen Fälle bei K und P liegen in anderen Geschwindigkeitsbereichen. Eine gemeinsame Grenze für alle Materialien wäre deshalb zu grob.
- **WDF:** Die Fehlerfälle zeigen unpassende Material-Bohrer-Kombinationen. Entscheidend ist die Abweichung von den passenden Zuordnungen K–H, P–N und N–W.

Die interaktive Auswahl macht Unterschiede zwischen den Materialgruppen deutlicher als die gemeinsame Betrachtung aller Beobachtungen. Die abgelesenen Werte sind vorläufige Anhaltspunkte; ihre Eignung als Fehlerregeln wird in Exercise 15.4 geprüft.

HTML erstellt mit:

uv run jupyter nbconvert --to html -- "Tasks/15 Self study - Session 2/Exercise_15.3.ipynb"